In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

"""
Herramientas Hidrogeológicas
Análisis de Pruebas de Bombeo (Método de Cooper-Jacob)

Descripción:
Ajuste semilogarítmico para estimar la Transmisividad (T) y el Coeficiente de Almacenamiento (S) mediante regresión lineal,
filtrando tiempos iniciales donde u > 0.03.

Autor: Carlos Javier Pérez Pérez
"""

In [ ]:
# 1. Datos de campo y parámetros iniciales
# Leer el archivo de Excel
df = pd.read_excel('datos_campo_jacob.xlsx')

caudal_lps: float = 20.0       
radio_obs_m: float = 150.0     

# Conversión automática de unidades
caudal_m3d: float = caudal_lps * 86.4 

In [ ]:
# 2. Filtrado de datos iniciales
minuto_inicio_analisis: int = 20  

df_valido = df[df['tiempo_min'] >= minuto_inicio_analisis].copy()
df_descartado = df[df['tiempo_min'] < minuto_inicio_analisis].copy()

In [ ]:
# 3. Cálculo Matemático (Regresión Lineal)
df_valido.loc[:, 'log_tiempo'] = np.log10(df_valido['tiempo_min'])

pendiente, intercepto, r_pearson, p_val, error_std = stats.linregress(
    df_valido['log_tiempo'], 
    df_valido['descenso_m']
)

# Delta s es equivalente a la pendiente en un ciclo logarítmico (diferencia log10 = 1)
delta_s: float = pendiente 

# Intersección con el eje de tiempo (cuando s = 0)
t0_min: float = 10 ** (-intercepto / pendiente)
t0_dias: float = t0_min / 1440.0 

# Fórmulas Clásicas de Cooper-Jacob
T_calc: float = (0.183 * caudal_m3d) / delta_s
S_calc: float = (2.25 * T_calc * t0_dias) / (radio_obs_m**2)
r_cuadrado: float = r_pearson**2

In [ ]:
# 4. Reporte en consola
print("Reporte Hidrogeológico: Método De Cooper-Jacob")
print("-" * 50)
print(f"Datos Analizados A Partir Del Minuto: {minuto_inicio_analisis}")
print(f"Calidad Del Ajuste (R²)             : {r_cuadrado:.4f}\n")
print(f"Transmisividad (T)                  : {T_calc:.1f} m²/día")
print(f"Almacenamiento (S)                  : {S_calc:.1e}")
print(f"Intersección De Tiempo (t0)         : {t0_min:.2f} min\n")

In [ ]:
# 5. Configuración y exportación del gráfico de diagnóstico
plt.rcParams.update({'font.family': 'serif', 'font.size': 11})
fig, ax = plt.subplots(figsize=(10, 6))

# Graficar datos descartados y válidos
ax.semilogx(df_descartado['tiempo_min'], df_descartado['descenso_m'], 
            'o', color='gray', alpha=0.5, markersize=8, label='Datos Ignorados ($u > 0.03$)')
ax.semilogx(df_valido['tiempo_min'], df_valido['descenso_m'], 
            's', color='#004c99', markersize=8, label='Datos De Análisis')

# Proyectar la recta desde t0 hasta el final
tiempos_recta = np.array([t0_min, df['tiempo_min'].max() * 1.2])
descensos_recta = pendiente * np.log10(tiempos_recta) + intercepto

ax.semilogx(tiempos_recta, descensos_recta, '--', color='#b30000', 
            linewidth=2, label=f'Ajuste Lineal ($R^2$ = {r_cuadrado:.3f})')

# Detalles estéticos del gráfico
ax.grid(True, which="both", ls="--", alpha=0.4, color='gray')
ax.invert_yaxis() # Invertir eje Y como en la representación clásica
ax.set_xlabel('Tiempo De Bombeo (Minutos) - Escala Logarítmica', fontweight='bold')
ax.set_ylabel('Descenso $s$ (Metros)', fontweight='bold')
ax.set_title('Análisis De Prueba De Bombeo: Método De Cooper-Jacob', pad=15, fontweight='bold')

# Caja de resultados integrada
texto_resultados = (
    f"Parámetros Estimados:\n"
    f"T = {T_calc:.1f} m²/d\n"
    f"S = {S_calc:.1e}"
)
props_caja = dict(boxstyle='square,pad=0.6', facecolor='#f9f9f9', edgecolor='black', alpha=0.9)

# Ubicación en la esquina inferior izquierda (para que no cubra los datos iniciales)
ax.text(0.04, 0.20, texto_resultados, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props_caja)

ax.legend(loc='lower left', framealpha=1.0, edgecolor='black')

# Guardar la imagen con alta resolución (300 DPI)
plt.savefig('grafico_jacob.png', dpi=300, bbox_inches='tight')

plt.tight_layout()
plt.show()